In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve, auc

In [2]:

#this is path to the dataset and the dataset is getting loaded using pandas
path = "/content/spambase.data"
df = pd.read_csv(path, header=None)

In [3]:

#these are all the columns from the spambase .names file
feat = ['word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d',
    'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet',
    'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free',
    'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money',
    'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650',
    'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857',
    'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology',
    'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project',
    'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$',
    'char_freq_#', 'capital_run_length_average', 'capital_run_length_longest',
    'capital_run_length_total', 'spam']

df.columns = feat
# we take all features in X and drop all columns except spam for y
X = df.drop('spam', axis=1).values
y = df['spam'].values

In [4]:
# 4.1

# function to split data into k folds
def split_into_folds(X, y, k):

    n = len(y)

    size = n // k

    # shuffle indices
    ind = np.arange(n)
    np.random.seed(42)
    np.random.shuffle(ind)

    # create folds
    li = []
    for i in range(k):
        s = i * size
        if i == k - 1:
            e = n
        else:
            e = s + size
        li.append(ind[s:e])

    return li





In [5]:
# function to get train and validation data for fold i
def get_fold_data(X, y, f, i):
    v_idx = f[i]
    t_idx = []
    for j in range(len(f)):
        if j != i:
            for idx in f[j]:
                t_idx.append(idx)

    X_train = X[t_idx]
    y_train = y[t_idx]
    X_val = X[v_idx]
    y_val = y[v_idx]

    return X_train, y_train, X_val, y_val



In [6]:
# function to run k-fold cv
def kfold_cv(X, y, model, k):
    folds = split_into_folds(X, y, k)
    errors = []

    for i in range(k):
        X_train, y_train, X_val, y_val = get_fold_data(X, y, folds, i)

        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        acc = accuracy_score(y_val, preds)
        err = 1 - acc
        errors.append(err)
        print("Fold " + str(i+1) + ": error = " + str(err))

    avg_err = sum(errors) / len(errors)
    print("Average error = " + str(avg_err))
    return avg_err

In [9]:
# 4.2
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# models
mod1 = LogisticRegression(max_iter=1000)
mod2 = LinearDiscriminantAnalysis()

# k values
k1 = 5
k2 = 10

print("Logistic Regression with k=5: ")
mod1_k5 = kfold_cv(X, y, mod1, k1)

print("\n")
print("Logistic Regression with k=10: ")
mod1_k10 = kfold_cv(X, y, mod1, k2)

print("\n")
print("LDA with k=5: ")
mod2_k5 = kfold_cv(X, y, mod2, k1)

print("\n")
print("LDA with k=10:")
mod2_k10 = kfold_cv(X, y, mod2, k2)

Logistic Regression with k=5: 
Fold 1: error = 0.08043478260869563
Fold 2: error = 0.07065217391304346
Fold 3: error = 0.07717391304347831
Fold 4: error = 0.06847826086956521
Fold 5: error = 0.07057546145494031
Average error = 0.07346291837794458


Logistic Regression with k=10: 
Fold 1: error = 0.07391304347826089
Fold 2: error = 0.07826086956521738
Fold 3: error = 0.05434782608695654
Fold 4: error = 0.07826086956521738
Fold 5: error = 0.08695652173913049
Fold 6: error = 0.05869565217391304
Fold 7: error = 0.07826086956521738
Fold 8: error = 0.05217391304347829
Fold 9: error = 0.06956521739130439
Fold 10: error = 0.07158351409978303
Average error = 0.07020182967084788


LDA with k=5: 
Fold 1: error = 0.11847826086956526
Fold 2: error = 0.10652173913043483
Fold 3: error = 0.11956521739130432
Fold 4: error = 0.10652173913043483
Fold 5: error = 0.1118349619978285
Average error = 0.11258438370391355


LDA with k=10:
Fold 1: error = 0.11739130434782608
Fold 2: error = 0.11304347826086958
F